<a id="top"></a>

# Demo: errors that close the loop, side effects that don't

```yaml
title: "Demo: errors that close the loop, side effects that don't"
keywords: unrecoverable error, recoverable error, found false, retry, idempotency, side effect, hidden side effect, send_message, lookup_user_by_email, tool_calls, ToolMessage, bind_tools, teacher demo
estimated duration: 15
```

> **Lesson:** L08. Teacher demo — Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. The longest demo (five sub-steps); builds on [L0807](L0807_lecture.ipynb). Clear outputs before committing.
>
> **The client is LangChain's `ChatAnthropic`** (from L03/L07). `bind_tools([fn])` registers a plain typed function as a tool — its definition inferred from the function's name, docstring, and type hints — the model's choice shows up in `AIMessage.tool_calls`, and results go back as a `ToolMessage`. The API key still loads through the config seam (`require_anthropic_key`); never hard-coded.
>
> The point to land: **the model treats an error as new context and will retry on its own when a result looks ambiguous — regardless of whether retry is safe.** Error *shape* decides recovery; a hidden side effect is one the model can't reason about, so it can't avoid it.

## Contents

- [1. Setup — a lookup tool, a side-effecting send, and a visible outbox](#1-setup--a-lookup-tool-a-side-effecting-send-and-a-visible-outbox)
- [2. Tool definitions — note the side effect is NOT named yet](#2-tool-definitions--note-the-side-effect-is-not-named-yet)
- [3. Unrecoverable error path — a clean {found: false}](#3-unrecoverable-error-path--a-clean-found-false)
- [4. Recoverable error path — a transient {error: lookup_failed}](#4-recoverable-error-path--a-transient-error-lookup_failed)
- [5. Hidden side effect — and naming it in the description](#5-hidden-side-effect--and-naming-it-in-the-description)
- [6. Non-idempotent retry — two identical sends](#6-non-idempotent-retry--two-identical-sends)
- [7. Takeaways](#7-takeaways)

## 1. Setup — a lookup tool, a side-effecting send, and a visible outbox

Three pieces: `lookup_user_by_email` (returns a clean `{'found': false}` for an unknown address, or a fake transient `{'error': 'lookup_failed'}` when a flake flag is on); `send_message` (logs to a **visible OUTBOX** and has **no idempotency key** — two calls send two messages); and `show_turn`, which prints each `AIMessage`'s text and `tool_calls`. Each is a plain typed function, so `bind_tools` turns it straight into a tool. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import require_anthropic_key

SONNET = "claude-sonnet-4-6"
model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)


KNOWN_USERS = {"alex@example.com": "u_alex"}  # priya is deliberately ABSENT
OUTBOX: list[dict[str, str]] = []  # every send_message lands here, visibly
CREATED: list[str] = []  # records the hidden-side-effect demo creates
SIMULATE_FLAKE = False  # flip on to force a transient lookup error


# bind_tools infers each tool from the function's name, docstring, and type hints --
# so a plain typed function IS the tool, and its docstring IS the description the model sees.
def lookup_user_by_email(
    email: Annotated[str, "exact email address."],
) -> dict[str, object]:
    """Look up a user by exact email. Returns {found: true, user_id} when the user exists,
    or {found: false} when no such user exists (a final answer -- do not retry).

    (Also returns a transient error when flaky.)"""
    if SIMULATE_FLAKE:
        return {"error": "lookup_failed"}  # recoverable: succeeds on retry
    if email in KNOWN_USERS:
        return {"found": True, "user_id": KNOWN_USERS[email]}
    return {"found": False, "email": email}  # clean, FINAL unrecoverable signal


def send_message(
    recipient_id: Annotated[str, "the recipient user_id."],
    body: Annotated[str, "the message text."],
) -> dict[str, object]:
    """Send a short message to a user by their user_id."""  # side effect HIDDEN
    OUTBOX.append({"to": recipient_id, "body": body})
    return {"sent": True, "outbox_size": len(OUTBOX)}


def show_turn(resp: AIMessage) -> None:
    print("tool_calls:", [c["name"] for c in resp.tool_calls])
    if resp.content:
        print("  text     ", repr(resp.content))
    for call in resp.tool_calls:
        print(f"  tool_call {call['name']}  args={call['args']}")


print("setup ready (live model:", SONNET, ")")

[↑ Back to top](#top)

## 2. Tool definitions — note the side effect is NOT named yet

Two side-effecting lookup variants for section 5. Both silently **create** a missing user; they differ only in their **docstring**, which is exactly the description `bind_tools` hands the model. `lookup_or_create` keeps an honest-looking description (hiding the create); `lookup_or_create_honest` names it. The `send_message` docstring in the setup cell also deliberately **omits** that it is non-idempotent — that's the side effect we'll reason about in section 6.

In [ ]:
# The two side-effecting lookup variants used in section 5. Both silently CREATE a missing
# user; they differ ONLY in the docstring -- and bind_tools reads the docstring as the
# model-facing description. That difference is the whole point of the section.
def _do_create(email: str) -> dict[str, object]:
    if email in KNOWN_USERS:
        return {"found": True, "user_id": KNOWN_USERS[email]}
    KNOWN_USERS[email] = f"u_{email.split('@')[0]}"  # <- the hidden write
    CREATED.append(email)
    return {"found": True, "user_id": KNOWN_USERS[email], "created": True}


def lookup_or_create(
    email: Annotated[str, "exact email address."],
) -> dict[str, object]:
    """Look up a user by their email."""  # side effect HIDDEN: silently creates a missing user
    return _do_create(email)


def lookup_or_create_honest(
    email: Annotated[str, "exact email address."],
) -> dict[str, object]:
    """Look up a user by email. If the user does NOT exist, this tool will CREATE one with
    default settings -- call only when a missing user should be auto-created."""
    return _do_create(email)


PROMPT = (
    "Look up the user with email priya@example.com, then send them a message saying "
    "the design review is at 2pm Tuesday."
)
print("tools defined")

[↑ Back to top](#top)

## 3. Unrecoverable error path — a clean {found: false}

Priya is **not** in the user table, and `SIMULATE_FLAKE` is off, so the lookup returns a clean `{'found': false}`. The model reads it as a **final** signal — it reports back or asks for the right email, and does **not** retry. Watch: no retry loop.

In [ ]:
SIMULATE_FLAKE = False
tooled = model.bind_tools([lookup_user_by_email, send_message])
dispatch = {"lookup_user_by_email": lookup_user_by_email, "send_message": send_message}
messages: list[BaseMessage] = [HumanMessage(PROMPT)]
for _ in range(4):
    resp = tooled.invoke(messages)
    show_turn(resp)
    if not resp.tool_calls:
        break
    messages.append(resp)
    for call in resp.tool_calls:
        out = dispatch[call["name"]](**call["args"])
        print("   ->", call["name"], "returned", out)
        messages.append(ToolMessage(content=str(out), tool_call_id=call["id"]))
print("\nOUTBOX:", OUTBOX, "(expect empty — no message sent to a nonexistent user)")

[↑ Back to top](#top)

## 4. Recoverable error path — a transient {error: lookup_failed}

Now flip `SIMULATE_FLAKE` on for the first lookup so it returns `{'error': 'lookup_failed'}`. The model usually **retries** the same call (sometimes twice). Worth asking yourself: who *should* retry here — the runtime (silently, with backoff) or the model (visibly, costing a turn)? Here the flake clears after the first call so a retry succeeds for `alex@example.com`.

In [ ]:
OUTBOX.clear()
flaky_prompt = "Look up alex@example.com and tell me their user_id."
tooled = model.bind_tools([lookup_user_by_email])
messages = [HumanMessage(flaky_prompt)]
SIMULATE_FLAKE = True  # first lookup will fail transiently
for _turn in range(4):
    resp = tooled.invoke(messages)
    show_turn(resp)
    if not resp.tool_calls:
        break
    SIMULATE_FLAKE = False  # the flake clears: a retry now succeeds
    messages.append(resp)
    for call in resp.tool_calls:
        out = lookup_user_by_email(**call["args"])
        print("   ->", call["name"], "returned", out)
        messages.append(ToolMessage(content=str(out), tool_call_id=call["id"]))

[↑ Back to top](#top)

## 5. Hidden side effect — and naming it in the description

Bind `lookup_or_create` (which silently **creates** a missing user) with its honest-*looking* description. Re-run for Priya: a user is created the prompt never asked for. Then bind `lookup_or_create_honest` — same behavior, but its docstring **names** the side effect — and re-run: the model now pauses to reason about it. A side effect not named in the description is one the model can't avoid.

In [ ]:
# Bind the BAD lookup whose description HIDES the create, alongside send_message.
CREATED.clear()
tooled = model.bind_tools([lookup_or_create, send_message])
resp = tooled.invoke([HumanMessage(PROMPT)])
print("=== hidden side effect (description unchanged) ===")
show_turn(resp)
for call in resp.tool_calls:
    if call["name"] == "lookup_or_create":
        print("   ->", lookup_or_create(**call["args"]))
print("CREATED (unintended!):", CREATED)

# Now bind the variant whose description NAMES the side effect and re-run.
CREATED.clear()
tooled = model.bind_tools([lookup_or_create_honest, send_message])
resp = tooled.invoke([HumanMessage(PROMPT)])
print("\n=== side effect NAMED in the description ===")
show_turn(resp)
print("(the model now often pauses to ask before creating a user)")

[↑ Back to top](#top)

## 6. Non-idempotent retry — two identical sends

Finally, the idempotency point — shown directly so it's deterministic. `send_message` has **no idempotency key**, so calling it twice with the same arguments sends **two** messages. A confused model that retries an ambiguous send produces a duplicate. The mitigation lives at the **runtime/design** layer, not in hoping the model behaves.

In [ ]:
OUTBOX.clear()
send_message("u_alex", "The design review is at 2pm Tuesday.")
send_message("u_alex", "The design review is at 2pm Tuesday.")  # model retried 'just in case'
print("OUTBOX after a retry:")
for i, m in enumerate(OUTBOX, start=1):
    print(f"  {i}. to={m['to']}  body={m['body']!r}")
print("two identical messages sent — no idempotency key to dedupe them.")

[↑ Back to top](#top)

## 7. Takeaways

- The **three error classes are interfaces**, not edge cases. `{found: false}` is a *final, unrecoverable* signal (no retry); `{error: lookup_failed}` is *recoverable* (retry); a validation error ([L0807](L0807_lecture.ipynb)) names the field to fix. The error *shape* drives the model's recovery.
- A **side effect not named in the description is one the model can't reason about** — and therefore can't avoid. Naming it is part of the contract.
- **Idempotency is a system concern.** The model *will* retry a non-idempotent call when a result looks ambiguous. Mitigate at the runtime: idempotency keys, a confirmation step, or a description that warns against retry.
- Forward link: **L11 (human-in-the-loop / approval gates)** revisits this exact tension for high-stakes side effects. The [L0810 lab](L0810_lab_empty.ipynb) has you classify errors and tag side effects, offline.
- Bridge to **L09 (MCP)**: every design choice in L08 — name, description, schema, error shape — *is* the MCP tool spec. MCP changes the transport, not the design.

[↑ Back to top](#top)